# Semana 4 — Cadenas absorbentes

En esta sesión estudiamos cadenas de Markov con **estados absorbentes**: una vez que el proceso entra en uno de ellos, no puede salir.

La matriz de transición tiene la forma canónica:

$$
P=
\begin{pmatrix}
\mathbf Q & \mathbf R\\
\mathbf 0 & \mathbf I
\end{pmatrix},
$$

donde:

- $\mathbf Q$ contiene las transiciones entre estados **transitorios**;
- $\mathbf R$ contiene las transiciones desde estados transitorios hacia estados **absorbentes**.

### Conexión con la semana 3

En la semana anterior calculamos ocupaciones durante un horizonte finito:

$$
\mathbf M(H)=\mathbf I+\mathbf P+\mathbf P^2+\cdots+\mathbf P^{H-1}.
$$

Ahora acumularemos la ocupación de los estados transitorios **hasta que ocurra la absorción**:

$$
\mathbf N
=
\mathbf I+\mathbf Q+\mathbf Q^2+\mathbf Q^3+\cdots
=
(\mathbf I-\mathbf Q)^{-1}.
$$

El elemento $N_{ij}$ representa el **número esperado de períodos que el proceso pasa en el estado transitorio $j$ antes de la absorción**, si comienza en el estado transitorio $i$.

> **Importante:** $N_{ij}$ no es un tiempo de primera pasada hacia $j$.  
> La suma de la fila $i$, $\sum_j N_{ij}$, sí representa el tiempo total esperado antes de la absorción.

Una vez calculada $\mathbf N$, las probabilidades de terminar en cada estado absorbente se obtienen mediante:

$$
\mathbf B=\mathbf N\mathbf R.
$$

Como en las semanas anteriores, primero calculamos con NumPy y al final utilizamos `jmarkov` para verificar.

### Cómo trabajar con este cuaderno

Encontrará tres tipos de código:

- 🧠 **Código central:** debe poder explicar qué representa cada instrucción.
- 🧩 **Herramienta nueva:** se introduce con una explicación breve.
- ✅ **Verificación:** se usa para comprobar resultados ya calculados.

Ejecute las celdas de arriba hacia abajo. Antes de comenzar, reinicie el kernel y ejecute todo el cuaderno.

In [1]:
!pip install jmarkov

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import numpy as np
from jmarkov.dtmc import dtmc

### 🧩 Herramienta nueva: la matriz identidad

La función `np.eye(n)` construye la matriz identidad de tamaño $n\times n$.

La necesitamos porque el primer término de la serie de la matriz fundamental es:

$$
\mathbf Q^0=\mathbf I.
$$

In [3]:
identidad_3 = np.eye(3)

print("Matriz identidad de tamaño 3:")
print(identidad_3)

Matriz identidad de tamaño 3:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


---
## Ejercicio 1 — Portafolio de proyectos de inversión *(básico)*

Un gestor de portafolios clasifica cada proyecto al cierre del año según su rentabilidad.

$$
X_n=\text{clasificación del proyecto al final del año }n
$$

Los estados son:

- **PR:** Poco rentable.
- **R:** Rentable.
- **MR:** Muy rentable.
- **NR:** No rentable, absorbente.
- **L:** Liquidado, absorbente.

La matriz de transición es:

$$
P=
\begin{pmatrix}
0.30&0.20&0.00&0.30&0.20\\
0.10&0.50&0.20&0.10&0.10\\
0.00&0.20&0.80&0.00&0.00\\
0.00&0.00&0.00&1.00&0.00\\
0.00&0.00&0.00&0.00&1.00
\end{pmatrix}.
$$

Los ingresos anuales son:

- $c_{PR}=\$38$ millones;
- $c_R=\$76$ millones;
- $c_{MR}=\$114$ millones.

Al producirse la absorción se incurre en un costo de salida:

- $c_{NR}=\$152$ millones;
- $c_L=\$190$ millones.

Para concentrarnos en cadenas absorbentes, **no aplicaremos descuento financiero**. Por tanto, los resultados representan ingresos y costos acumulados esperados sin valor temporal del dinero.

### a) Construcción y verificación de la matriz

Construya la matriz $P$, identifique los estados transitorios y absorbentes y verifique que cada fila suma 1.

In [4]:
estados_proyecto = ["PR", "R", "MR", "NR", "L"]

numero_transitorios_proyecto = 3
numero_absorbentes_proyecto = 2

P_proyecto = np.array([
    [0.30, 0.20, 0.00, 0.30, 0.20],
    [0.10, 0.50, 0.20, 0.10, 0.10],
    [0.00, 0.20, 0.80, 0.00, 0.00],
    [0.00, 0.00, 0.00, 1.00, 0.00],
    [0.00, 0.00, 0.00, 0.00, 1.00]
])

for indice_estado in range(len(estados_proyecto)):
    suma_fila = P_proyecto[indice_estado, :].sum()

    print(
        "Fila",
        estados_proyecto[indice_estado],
        "- suma:",
        round(suma_fila, 2)
    )

print()
print("Estados transitorios:", estados_proyecto[:3])
print("Estados absorbentes:", estados_proyecto[3:])

Fila PR - suma: 1.0
Fila R - suma: 1.0
Fila MR - suma: 1.0
Fila NR - suma: 1.0
Fila L - suma: 1.0

Estados transitorios: ['PR', 'R', 'MR']
Estados absorbentes: ['NR', 'L']


### b) Matriz fundamental

Separe los bloques $\mathbf Q$ y $\mathbf R$.

Primero calcularemos $\mathbf Q^2$ término a término:

$$
[\mathbf Q^2]_{ij}
=
\sum_k Q_{ik}Q_{kj}.
$$

Después acumularemos:

$$
\mathbf N=\mathbf I+\mathbf Q+\mathbf Q^2+\cdots
$$

hasta que el término $\mathbf Q^n$ sea despreciable.

In [5]:
Q_proyecto = P_proyecto[:3, :3]
R_proyecto = P_proyecto[:3, 3:]

Q2_termino_a_termino = np.zeros((3, 3))

for fila in range(3):
    for columna in range(3):

        suma = 0.0

        for estado_intermedio in range(3):
            suma = (
                suma
                + Q_proyecto[fila, estado_intermedio]
                * Q_proyecto[estado_intermedio, columna]
            )

        Q2_termino_a_termino[fila, columna] = suma

print("Q^2 calculada término a término:")
print(np.round(Q2_termino_a_termino, 4))

print()
print("Q^2 calculada con Q @ Q:")
print(np.round(Q_proyecto @ Q_proyecto, 4))

Q^2 calculada término a término:
[[0.11 0.16 0.04]
 [0.08 0.31 0.26]
 [0.02 0.26 0.68]]

Q^2 calculada con Q @ Q:
[[0.11 0.16 0.04]
 [0.08 0.31 0.26]
 [0.02 0.26 0.68]]


In [6]:
numero_transitorios = 3
tolerancia = 0.0000000001
maximo_iteraciones = 2000

matriz_fundamental_proyecto = np.eye(numero_transitorios)
potencia_Q = np.eye(numero_transitorios)

for iteracion in range(1, maximo_iteraciones + 1):

    potencia_Q = potencia_Q @ Q_proyecto

    matriz_fundamental_proyecto = (
        matriz_fundamental_proyecto + potencia_Q
    )

    cambio_maximo = np.max(np.abs(potencia_Q))

    if cambio_maximo < tolerancia:
        break

print("Iteraciones hasta convergencia:", iteracion)
print()
print("Matriz fundamental N:")
print(np.round(matriz_fundamental_proyecto, 4))

Iteraciones hasta convergencia: 234

Matriz fundamental N:
[[1.5789 1.0526 1.0526]
 [0.5263 3.6842 3.6842]
 [0.5263 3.6842 8.6842]]


In [7]:
indice_PR = 0
indice_R = 1
indice_MR = 2

print("Ocupaciones esperadas antes de la absorción desde MR:")

for estado_visitado in range(3):

    ocupacion = matriz_fundamental_proyecto[
        indice_MR,
        estado_visitado
    ]

    print(
        "Estado",
        estados_proyecto[estado_visitado],
        ":",
        round(ocupacion, 4),
        "años"
    )

tiempo_total_MR = matriz_fundamental_proyecto[
    indice_MR,
    :
].sum()

tiempo_total_PR = matriz_fundamental_proyecto[
    indice_PR,
    :
].sum()

print()
print(
    "Tiempo total esperado en el portafolio desde MR:",
    round(tiempo_total_MR, 2),
    "años"
)
print(
    "Tiempo total esperado en el portafolio desde PR:",
    round(tiempo_total_PR, 2),
    "años"
)
print(
    "Razón MR/PR:",
    round(tiempo_total_MR / tiempo_total_PR, 2)
)

Ocupaciones esperadas antes de la absorción desde MR:
Estado PR : 0.5263 años
Estado R : 3.6842 años
Estado MR : 8.6842 años

Tiempo total esperado en el portafolio desde MR: 12.89 años
Tiempo total esperado en el portafolio desde PR: 3.68 años
Razón MR/PR: 3.5


### c) Probabilidades de absorción

Calcule:

$$
\mathbf B=\mathbf N\mathbf R.
$$

Cada fila de $\mathbf B$ debe sumar 1 porque el proyecto termina finalmente en **NR** o en **L**.

In [8]:
probabilidades_absorcion_proyecto = (
    matriz_fundamental_proyecto @ R_proyecto
)

print("Probabilidades de absorción:")

for estado_inicial in range(3):

    probabilidad_NR = probabilidades_absorcion_proyecto[
        estado_inicial,
        0
    ]

    probabilidad_L = probabilidades_absorcion_proyecto[
        estado_inicial,
        1
    ]

    suma_probabilidades = probabilidad_NR + probabilidad_L

    print()
    print("Inicio:", estados_proyecto[estado_inicial])
    print("P(NR):", round(probabilidad_NR, 4))
    print("P(L):", round(probabilidad_L, 4))
    print("Suma:", round(suma_probabilidades, 4))

    if abs(suma_probabilidades - 1.0) > 0.000001:
        print("Advertencia: las probabilidades no suman 1.")

Probabilidades de absorción:

Inicio: PR
P(NR): 0.5789
P(L): 0.4211
Suma: 1.0

Inicio: R
P(NR): 0.5263
P(L): 0.4737
Suma: 1.0

Inicio: MR
P(NR): 0.5263
P(L): 0.4737
Suma: 1.0


### d) Ganancia neta acumulada esperada

Para un proyecto que inicia en **MR**:

$$
E[\text{ganancia}\mid MR]
=
\sum_j c_jN_{MR,j}
-
\sum_k c_kB_{MR,k}.
$$

La matriz $\mathbf N$ incluye la ocupación del estado inicial. Por tanto, suponemos que el proyecto genera el ingreso correspondiente a su clasificación inicial.

In [9]:
ingresos_por_estado = [38, 76, 114]
costos_de_salida = [152, 190]

ingresos_esperados = 0.0

for estado_transitorio in range(3):

    ingreso_estado = ingresos_por_estado[estado_transitorio]

    ocupacion_estado = matriz_fundamental_proyecto[
        indice_MR,
        estado_transitorio
    ]

    ingresos_esperados = (
        ingresos_esperados
        + ingreso_estado * ocupacion_estado
    )

costos_esperados_salida = 0.0

for estado_absorbente in range(2):

    costo_estado = costos_de_salida[estado_absorbente]

    probabilidad_estado = probabilidades_absorcion_proyecto[
        indice_MR,
        estado_absorbente
    ]

    costos_esperados_salida = (
        costos_esperados_salida
        + costo_estado * probabilidad_estado
    )

ganancia_neta_esperada = (
    ingresos_esperados - costos_esperados_salida
)

print(
    "Ingresos acumulados esperados:",
    round(ingresos_esperados, 2),
    "millones de pesos"
)
print(
    "Costos esperados de salida:",
    round(costos_esperados_salida, 2),
    "millones de pesos"
)
print(
    "Ganancia neta acumulada esperada:",
    round(ganancia_neta_esperada, 2),
    "millones de pesos"
)

Ingresos acumulados esperados: 1290.0 millones de pesos
Costos esperados de salida: 170.0 millones de pesos
Ganancia neta acumulada esperada: 1120.0 millones de pesos


### ✅ Verificación con `jmarkov`

Construimos las matrices de verificación mediante ciclos explícitos.

In [10]:
cadena_proyecto = dtmc(P_proyecto)

N_proyecto_jmarkov = np.zeros((3, 3))

for estado_inicial in range(3):
    for estado_visitado in range(3):

        valor = cadena_proyecto.absorbtion_times(
            start=estado_inicial,
            target=estado_visitado
        )

        N_proyecto_jmarkov[
            estado_inicial,
            estado_visitado
        ] = float(valor)

B_proyecto_jmarkov = np.zeros((3, 2))

for estado_inicial in range(3):
    for estado_absorbente in range(2):

        indice_absorbente = 3 + estado_absorbente

        valor = cadena_proyecto.absorbtion_probabilities(
            start=estado_inicial,
            target=indice_absorbente
        )

        B_proyecto_jmarkov[
            estado_inicial,
            estado_absorbente
        ] = float(valor[0][0])

print("N calculada con NumPy:")
print(np.round(matriz_fundamental_proyecto, 4))

print()
print("N calculada con jmarkov:")
print(np.round(N_proyecto_jmarkov, 4))

print()
print("B calculada con NumPy:")
print(np.round(probabilidades_absorcion_proyecto, 4))

print()
print("B calculada con jmarkov:")
print(np.round(B_proyecto_jmarkov, 4))

N calculada con NumPy:
[[1.5789 1.0526 1.0526]
 [0.5263 3.6842 3.6842]
 [0.5263 3.6842 8.6842]]

N calculada con jmarkov:
[[1.5789 1.0526 1.0526]
 [0.5263 3.6842 3.6842]
 [0.5263 3.6842 8.6842]]

B calculada con NumPy:
[[0.5789 0.4211]
 [0.5263 0.4737]
 [0.5263 0.4737]]

B calculada con jmarkov:
[[0.5789 0.4211]
 [0.5263 0.4737]
 [0.5263 0.4737]]


### 🧩 Reutilización del procedimiento

En los ejercicios siguientes repetiremos varias veces el cálculo de:

$$
\mathbf N=\mathbf I+\mathbf Q+\mathbf Q^2+\cdots
$$

Como ya desarrollamos el algoritmo completo, ahora lo reunimos en una función. Las funciones fueron trabajadas en la semana 3 y permiten evitar copiar el mismo bloque varias veces.

In [11]:
def calcular_matriz_fundamental(
    Q,
    tolerancia,
    maximo_iteraciones
):
    numero_estados = Q.shape[0]

    matriz_fundamental = np.eye(numero_estados)
    potencia_Q = np.eye(numero_estados)

    for iteracion in range(1, maximo_iteraciones + 1):

        potencia_Q = potencia_Q @ Q

        matriz_fundamental = (
            matriz_fundamental + potencia_Q
        )

        cambio_maximo = np.max(np.abs(potencia_Q))

        if cambio_maximo < tolerancia:
            break

    return matriz_fundamental, iteracion

---
## Ejercicio 2 — Ciclo de vida de una turbina industrial *(intermedio)*

Una empresa energética monitorea semanalmente una turbina.

Los estados son:

- **O:** Operativo.
- **DM:** Desgaste menor.
- **DG:** Desgaste grave.
- **F:** Falla irreparable, absorbente.

$$
P=
\begin{pmatrix}
0.85&0.13&0.02&0.00\\
0.00&0.65&0.30&0.05\\
0.00&0.00&0.50&0.50\\
0.00&0.00&0.00&1.00
\end{pmatrix}.
$$

Costos semanales de mantenimiento, en miles de COP:

- $c_O=500$;
- $c_{DM}=1200$;
- $c_{DG}=3000$.

El costo de reemplazo al fallar es $c_F=20000$ miles de COP.

### a) Construcción y verificación

In [12]:
estados_turbina = ["O", "DM", "DG", "F"]

P_turbina = np.array([
    [0.85, 0.13, 0.02, 0.00],
    [0.00, 0.65, 0.30, 0.05],
    [0.00, 0.00, 0.50, 0.50],
    [0.00, 0.00, 0.00, 1.00]
])

for indice_estado in range(len(estados_turbina)):

    suma_fila = P_turbina[indice_estado, :].sum()

    print(
        "Fila",
        estados_turbina[indice_estado],
        "- suma:",
        round(suma_fila, 2)
    )

print()
print("F es el único estado absorbente.")

Fila O - suma: 1.0
Fila DM - suma: 1.0
Fila DG - suma: 1.0
Fila F - suma: 1.0

F es el único estado absorbente.


### b) Matriz fundamental y vida útil

Calcule la matriz fundamental y recuerde:

- $N_{ij}$: semanas esperadas en el estado $j$ antes de fallar, comenzando en $i$;
- $\sum_jN_{ij}$: vida útil esperada antes de fallar.

In [13]:
Q_turbina = P_turbina[:3, :3]
R_turbina = P_turbina[:3, 3:]

matriz_fundamental_turbina, iteraciones_turbina = (
    calcular_matriz_fundamental(
        Q_turbina,
        0.0000000001,
        2000
    )
)

print(
    "Iteraciones hasta convergencia:",
    iteraciones_turbina
)
print()
print("Matriz fundamental de la turbina:")
print(np.round(matriz_fundamental_turbina, 3))

Iteraciones hasta convergencia: 142

Matriz fundamental de la turbina:
[[6.667 2.476 1.752]
 [0.    2.857 1.714]
 [0.    0.    2.   ]]


In [14]:
print("Vida útil esperada desde cada estado:")

for estado_inicial in range(3):

    vida_util = matriz_fundamental_turbina[
        estado_inicial,
        :
    ].sum()

    print()
    print("Estado inicial:", estados_turbina[estado_inicial])
    print("Semanas:", round(vida_util, 2))
    print("Años aproximados:", round(vida_util / 52, 2))

Vida útil esperada desde cada estado:

Estado inicial: O
Semanas: 10.9
Años aproximados: 0.21

Estado inicial: DM
Semanas: 4.57
Años aproximados: 0.09

Estado inicial: DG
Semanas: 2.0
Años aproximados: 0.04


### c) Probabilidad de falla y costo total esperado

Como **F** es el único estado absorbente, la probabilidad de terminar en F debe ser 1 desde cualquier estado transitorio.

El costo total esperado incluye:

1. los costos semanales acumulados en los estados transitorios;
2. el costo de reemplazo, pagado una sola vez cuando ocurre la falla.

In [15]:
probabilidades_absorcion_turbina = (
    matriz_fundamental_turbina @ R_turbina
)

for estado_inicial in range(3):

    probabilidad_falla = probabilidades_absorcion_turbina[
        estado_inicial,
        0
    ]

    print(
        "P(Falla | inicio",
        estados_turbina[estado_inicial],
        ") =",
        round(probabilidad_falla, 6)
    )

P(Falla | inicio O ) = 1.0
P(Falla | inicio DM ) = 1.0
P(Falla | inicio DG ) = 1.0


In [16]:
costos_mantenimiento = [500, 1200, 3000]
costo_reemplazo = 20000

costos_totales_turbina = np.zeros(3)

print("Costo total esperado, en miles de COP:")

for estado_inicial in range(3):

    costo_mantenimiento_esperado = 0.0

    for estado_visitado in range(3):

        costo_mantenimiento_esperado = (
            costo_mantenimiento_esperado
            + costos_mantenimiento[estado_visitado]
            * matriz_fundamental_turbina[
                estado_inicial,
                estado_visitado
            ]
        )

    costo_total = (
        costo_mantenimiento_esperado + costo_reemplazo
    )

    costos_totales_turbina[estado_inicial] = costo_total

    print()
    print("Estado inicial:", estados_turbina[estado_inicial])
    print(
        "Mantenimiento:",
        round(costo_mantenimiento_esperado, 2)
    )
    print("Reemplazo:", costo_reemplazo)
    print("Costo total:", round(costo_total, 2))

Costo total esperado, en miles de COP:

Estado inicial: O
Mantenimiento: 11561.9
Reemplazo: 20000
Costo total: 31561.9

Estado inicial: DM
Mantenimiento: 8571.43
Reemplazo: 20000
Costo total: 28571.43

Estado inicial: DG
Mantenimiento: 6000.0
Reemplazo: 20000
Costo total: 26000.0


### d) Programa de mantenimiento preventivo

El programa modifica la fila de **DM**:

$$
P'_{DM}=[0.10,\;0.65,\;0.20,\;0.05].
$$

Compare desde **O**:

- la vida útil esperada;
- el costo acumulado esperado.

> Una vida útil mayor no implica automáticamente que el programa sea económicamente conveniente. Para decidirlo también necesitaríamos el valor de las semanas adicionales de operación y el costo directo del programa preventivo.

In [17]:
P_turbina_preventivo = P_turbina.copy()

P_turbina_preventivo[1, :] = [
    0.10,
    0.65,
    0.20,
    0.05
]

Q_turbina_preventivo = P_turbina_preventivo[:3, :3]

matriz_fundamental_preventivo, iteraciones_preventivo = (
    calcular_matriz_fundamental(
        Q_turbina_preventivo,
        0.0000000001,
        2000
    )
)

vida_original = matriz_fundamental_turbina[0, :].sum()
vida_preventiva = matriz_fundamental_preventivo[0, :].sum()

costo_original = costos_totales_turbina[0]

costo_mantenimiento_preventivo = 0.0

for estado_visitado in range(3):

    costo_mantenimiento_preventivo = (
        costo_mantenimiento_preventivo
        + costos_mantenimiento[estado_visitado]
        * matriz_fundamental_preventivo[
            0,
            estado_visitado
        ]
    )

costo_preventivo = (
    costo_mantenimiento_preventivo + costo_reemplazo
)

print("Comparación desde el estado O:")
print()
print("Vida útil original:", round(vida_original, 2), "semanas")
print(
    "Vida útil con preventivo:",
    round(vida_preventiva, 2),
    "semanas"
)
print(
    "Incremento de vida útil:",
    round(vida_preventiva - vida_original, 2),
    "semanas"
)

print()
print(
    "Costo acumulado original:",
    round(costo_original, 2),
    "miles de COP"
)
print(
    "Costo acumulado con preventivo:",
    round(costo_preventivo, 2),
    "miles de COP"
)
print(
    "Incremento del costo acumulado:",
    round(costo_preventivo - costo_original, 2),
    "miles de COP"
)

Comparación desde el estado O:

Vida útil original: 10.9 semanas
Vida útil con preventivo: 13.82 semanas
Incremento de vida útil: 2.93 semanas

Costo acumulado original: 31561.9 miles de COP
Costo acumulado con preventivo: 33392.41 miles de COP
Incremento del costo acumulado: 1830.5 miles de COP


### ✅ Verificación con `jmarkov`

In [18]:
cadena_turbina = dtmc(P_turbina)
cadena_turbina_preventivo = dtmc(P_turbina_preventivo)

N_turbina_jmarkov = np.zeros((3, 3))
N_preventivo_jmarkov = np.zeros((3, 3))

for estado_inicial in range(3):
    for estado_visitado in range(3):

        valor_original = cadena_turbina.absorbtion_times(
            start=estado_inicial,
            target=estado_visitado
        )

        valor_preventivo = (
            cadena_turbina_preventivo.absorbtion_times(
                start=estado_inicial,
                target=estado_visitado
            )
        )

        N_turbina_jmarkov[
            estado_inicial,
            estado_visitado
        ] = float(valor_original)

        N_preventivo_jmarkov[
            estado_inicial,
            estado_visitado
        ] = float(valor_preventivo)

print("N original con NumPy:")
print(np.round(matriz_fundamental_turbina, 3))

print()
print("N original con jmarkov:")
print(np.round(N_turbina_jmarkov, 3))

print()
print("N preventivo con NumPy:")
print(np.round(matriz_fundamental_preventivo, 3))

print()
print("N preventivo con jmarkov:")
print(np.round(N_preventivo_jmarkov, 3))

N original con NumPy:
[[6.667 2.476 1.752]
 [0.    2.857 1.714]
 [0.    0.    2.   ]]

N original con jmarkov:
[[6.667 2.476 1.752]
 [0.    2.857 1.714]
 [0.    0.    2.   ]]

N preventivo con NumPy:
[[8.861 3.291 1.671]
 [2.532 3.797 1.62 ]
 [0.    0.    2.   ]]

N preventivo con jmarkov:
[[8.861 3.291 1.671]
 [2.532 3.797 1.62 ]
 [0.    0.    2.   ]]


---
## Ejercicio 3 — Bono corporativo con ciclo económico *(profundización guiada)*

Un bono corporativo evoluciona trimestralmente según:

1. su **rating crediticio**: AAA, BBB o CCC;
2. el **ciclo económico**: Expansión o Recesión.

El bono puede terminar en:

- **Default**, absorbente;
- **Prepago**, absorbente.

Los seis estados transitorios son:

| Índice | Estado |
|---:|---|
| 0 | (AAA, Exp) |
| 1 | (AAA, Rec) |
| 2 | (BBB, Exp) |
| 3 | (BBB, Rec) |
| 4 | (CCC, Exp) |
| 5 | (CCC, Rec) |

Los estados 6 y 7 son **Default** y **Prepago**.

La probabilidad de pasar de $(r,e)$ a $(r',e')$ es:

$$
(1-p_m)\,\alpha(r,e,r')\,q(e,e').
$$

La probabilidad de default es:

$$
(1-p_m)\,\alpha(r,e,\text{Default}),
$$

y la probabilidad de prepago es $p_m$.

### a) Datos del modelo y construcción de una fila

Primero representamos la migración de rating mediante una matriz de seis filas. Cada fila corresponde directamente a uno de los seis estados conjuntos.

Las columnas son:

$$
[\text{AAA},\text{BBB},\text{CCC},\text{Default}].
$$

Después construiremos como ejemplo la fila correspondiente a **(BBB, Rec)**.

In [19]:
estados_bono = [
    "(AAA, Exp)",
    "(AAA, Rec)",
    "(BBB, Exp)",
    "(BBB, Rec)",
    "(CCC, Exp)",
    "(CCC, Rec)",
    "Default",
    "Prepago"
]

transicion_economica = np.array([
    [0.85, 0.15],
    [0.35, 0.65]
])

migracion_rating = np.array([
    [0.92, 0.07, 0.01, 0.00],
    [0.85, 0.12, 0.02, 0.01],
    [0.04, 0.85, 0.10, 0.01],
    [0.02, 0.75, 0.18, 0.05],
    [0.01, 0.08, 0.80, 0.11],
    [0.00, 0.03, 0.60, 0.37]
])

probabilidad_prepago = [
    0.04,
    0.04,
    0.02,
    0.02,
    0.01,
    0.01
]

ciclo_del_estado = [
    0,
    1,
    0,
    1,
    0,
    1
]

In [20]:
# Ejemplo: construcción de la fila de (BBB, Rec)

estado_actual = 3
ciclo_actual = ciclo_del_estado[estado_actual]

factor_no_prepago = (
    1 - probabilidad_prepago[estado_actual]
)

fila_BBB_Rec = np.zeros(8)

for rating_siguiente in range(3):
    for ciclo_siguiente in range(2):

        estado_siguiente = (
            rating_siguiente * 2 + ciclo_siguiente
        )

        fila_BBB_Rec[estado_siguiente] = (
            factor_no_prepago
            * migracion_rating[
                estado_actual,
                rating_siguiente
            ]
            * transicion_economica[
                ciclo_actual,
                ciclo_siguiente
            ]
        )

fila_BBB_Rec[6] = (
    factor_no_prepago
    * migracion_rating[estado_actual, 3]
)

fila_BBB_Rec[7] = (
    probabilidad_prepago[estado_actual]
)

print("Fila correspondiente a (BBB, Rec):")
print(np.round(fila_BBB_Rec, 5))
print("Suma de la fila:", round(fila_BBB_Rec.sum(), 5))

Fila correspondiente a (BBB, Rec):
[0.00686 0.01274 0.25725 0.47775 0.06174 0.11466 0.049   0.02   ]
Suma de la fila: 1.0


### 🧩 Generalización de la construcción

La siguiente función repite exactamente el procedimiento anterior para los seis estados transitorios. Esta celda se desarrolla paso a paso en clase.

In [21]:
def construir_matriz_bono(
    matriz_ciclo,
    migracion,
    probabilidades_prepago,
    ciclos_actuales
):
    P = np.zeros((8, 8))

    for estado_actual in range(6):

        ciclo_actual = ciclos_actuales[estado_actual]

        factor_no_prepago = (
            1 - probabilidades_prepago[estado_actual]
        )

        for rating_siguiente in range(3):
            for ciclo_siguiente in range(2):

                estado_siguiente = (
                    rating_siguiente * 2
                    + ciclo_siguiente
                )

                P[estado_actual, estado_siguiente] = (
                    factor_no_prepago
                    * migracion[
                        estado_actual,
                        rating_siguiente
                    ]
                    * matriz_ciclo[
                        ciclo_actual,
                        ciclo_siguiente
                    ]
                )

        P[estado_actual, 6] = (
            factor_no_prepago
            * migracion[estado_actual, 3]
        )

        P[estado_actual, 7] = (
            probabilidades_prepago[estado_actual]
        )

    P[6, 6] = 1.0
    P[7, 7] = 1.0

    return P

In [22]:
P_bono = construir_matriz_bono(
    transicion_economica,
    migracion_rating,
    probabilidad_prepago,
    ciclo_del_estado
)

print("Dimensión de P:", P_bono.shape)
print()

for estado_actual in range(8):

    suma_fila = P_bono[estado_actual, :].sum()

    print(
        "Fila",
        estados_bono[estado_actual],
        "- suma:",
        round(suma_fila, 6)
    )

Dimensión de P: (8, 8)

Fila (AAA, Exp) - suma: 1.0
Fila (AAA, Rec) - suma: 1.0
Fila (BBB, Exp) - suma: 1.0
Fila (BBB, Rec) - suma: 1.0
Fila (CCC, Exp) - suma: 1.0
Fila (CCC, Rec) - suma: 1.0
Fila Default - suma: 1.0
Fila Prepago - suma: 1.0


### b) Matriz fundamental

Calcule $\mathbf N$ para los seis estados transitorios y analice un bono que comienza en **(BBB, Rec)**, cuyo índice es 3.

In [23]:
Q_bono = P_bono[:6, :6]
R_bono = P_bono[:6, 6:]

matriz_fundamental_bono, iteraciones_bono = (
    calcular_matriz_fundamental(
        Q_bono,
        0.0000000001,
        5000
    )
)

indice_BBB_Rec = 3

print("Iteraciones hasta convergencia:", iteraciones_bono)
print()
print(
    "Ocupaciones esperadas antes de la absorción",
    "desde (BBB, Rec):"
)

for estado_visitado in range(6):

    ocupacion = matriz_fundamental_bono[
        indice_BBB_Rec,
        estado_visitado
    ]

    print(
        estados_bono[estado_visitado],
        ":",
        round(ocupacion, 4),
        "trimestres"
    )

tiempo_total_BBB_Rec = matriz_fundamental_bono[
    indice_BBB_Rec,
    :
].sum()

print()
print(
    "Tiempo total antes de la absorción:",
    round(tiempo_total_BBB_Rec, 2),
    "trimestres"
)
print(
    "Años aproximados:",
    round(tiempo_total_BBB_Rec / 4, 2)
)

Iteraciones hasta convergencia: 251

Ocupaciones esperadas antes de la absorción desde (BBB, Rec):
(AAA, Exp) : 1.0929 trimestres
(AAA, Rec) : 0.4348 trimestres
(BBB, Exp) : 3.3299 trimestres
(BBB, Rec) : 2.8805 trimestres
(CCC, Exp) : 2.0863 trimestres
(CCC, Rec) : 1.0329 trimestres

Tiempo total antes de la absorción: 10.86 trimestres
Años aproximados: 2.71


### c) Tiempo total desde cada estado

Compare los estados de expansión y recesión para un mismo rating.

In [24]:
print("Tiempo total esperado antes de la absorción:")

for estado_inicial in range(6):

    tiempo_total = matriz_fundamental_bono[
        estado_inicial,
        :
    ].sum()

    print()
    print("Estado:", estados_bono[estado_inicial])
    print("Trimestres:", round(tiempo_total, 2))
    print("Años aproximados:", round(tiempo_total / 4, 2))

Tiempo total esperado antes de la absorción:

Estado: (AAA, Exp)
Trimestres: 15.44
Años aproximados: 3.86

Estado: (AAA, Rec)
Trimestres: 14.6
Años aproximados: 3.65

Estado: (BBB, Exp)
Trimestres: 12.77
Años aproximados: 3.19

Estado: (BBB, Rec)
Trimestres: 10.86
Años aproximados: 2.71

Estado: (CCC, Exp)
Trimestres: 8.38
Años aproximados: 2.09

Estado: (CCC, Rec)
Trimestres: 5.02
Años aproximados: 1.26


### d) Cupones acumulados esperados

El bono paga por trimestre:

- AAA: 0.03 del principal;
- BBB: 0.06 del principal;
- CCC: 0.12 del principal.

Calcularemos cupones acumulados esperados **sin descuento financiero**.

In [25]:
cupon_por_estado = [
    0.03,
    0.03,
    0.06,
    0.06,
    0.12,
    0.12
]

cupones_acumulados = np.zeros(6)

for estado_inicial in range(6):

    cupon_esperado = 0.0

    for estado_visitado in range(6):

        cupon_esperado = (
            cupon_esperado
            + cupon_por_estado[estado_visitado]
            * matriz_fundamental_bono[
                estado_inicial,
                estado_visitado
            ]
        )

    cupones_acumulados[estado_inicial] = (
        cupon_esperado
    )

    print(
        "Inicio",
        estados_bono[estado_inicial],
        "- cupón acumulado esperado:",
        round(cupon_esperado, 4)
    )

Inicio (AAA, Exp) - cupón acumulado esperado: 0.8029
Inicio (AAA, Rec) - cupón acumulado esperado: 0.7807
Inicio (BBB, Exp) - cupón acumulado esperado: 0.9026
Inicio (BBB, Rec) - cupón acumulado esperado: 0.7928
Inicio (CCC, Exp) - cupón acumulado esperado: 0.789
Inicio (CCC, Rec) - cupón acumulado esperado: 0.5029


### e) Probabilidades de absorción

Calcule:

$$
\mathbf B=\mathbf N\mathbf R.
$$

Interprete por separado la probabilidad de **Default** y la probabilidad de **Prepago**.

In [26]:
probabilidades_absorcion_bono = (
    matriz_fundamental_bono @ R_bono
)

for estado_inicial in range(6):

    probabilidad_default = probabilidades_absorcion_bono[
        estado_inicial,
        0
    ]

    probabilidad_prepago_estado = (
        probabilidades_absorcion_bono[
            estado_inicial,
            1
        ]
    )

    suma = (
        probabilidad_default
        + probabilidad_prepago_estado
    )

    print()
    print("Inicio:", estados_bono[estado_inicial])
    print(
        "P(Default):",
        round(probabilidad_default, 4)
    )
    print(
        "P(Prepago):",
        round(probabilidad_prepago_estado, 4)
    )
    print("Suma:", round(suma, 4))


Inicio: (AAA, Exp)
P(Default): 0.5387
P(Prepago): 0.4613
Suma: 1.0

Inicio: (AAA, Rec)
P(Default): 0.5732
P(Prepago): 0.4268
Suma: 1.0

Inicio: (BBB, Exp)
P(Default): 0.736
P(Prepago): 0.264
Suma: 1.0

Inicio: (BBB, Rec)
P(Default): 0.7835
P(Prepago): 0.2165
Suma: 1.0

Inicio: (CCC, Exp)
P(Default): 0.8669
P(Prepago): 0.1331
Suma: 1.0

Inicio: (CCC, Rec)
P(Default): 0.9274
P(Prepago): 0.0726
Suma: 1.0


### f) Sensibilidad: economía siempre en expansión

Ahora suponemos:

$$
q'(Exp\to Exp)=1,
\qquad
q'(Rec\to Exp)=1.
$$

Reutilice las funciones anteriores y compare desde **(BBB, Rec)**:

- la probabilidad de default;
- el tiempo total esperado antes de la absorción.

In [27]:
economia_siempre_expansion = np.array([
    [1.0, 0.0],
    [1.0, 0.0]
])

P_bono_expansion = construir_matriz_bono(
    economia_siempre_expansion,
    migracion_rating,
    probabilidad_prepago,
    ciclo_del_estado
)

Q_bono_expansion = P_bono_expansion[:6, :6]
R_bono_expansion = P_bono_expansion[:6, 6:]

matriz_fundamental_expansion, iteraciones_expansion = (
    calcular_matriz_fundamental(
        Q_bono_expansion,
        0.00000001,
        20000
    )
)

probabilidades_absorcion_expansion = (
    matriz_fundamental_expansion
    @ R_bono_expansion
)

probabilidad_default_original = (
    probabilidades_absorcion_bono[
        indice_BBB_Rec,
        0
    ]
)

probabilidad_default_expansion = (
    probabilidades_absorcion_expansion[
        indice_BBB_Rec,
        0
    ]
)

tiempo_original = matriz_fundamental_bono[
    indice_BBB_Rec,
    :
].sum()

tiempo_expansion = matriz_fundamental_expansion[
    indice_BBB_Rec,
    :
].sum()

print("Comparación desde (BBB, Rec):")
print()
print(
    "P(Default), economía mixta:",
    round(probabilidad_default_original, 4)
)
print(
    "P(Default), solo expansión:",
    round(probabilidad_default_expansion, 4)
)
print(
    "Tiempo, economía mixta:",
    round(tiempo_original, 2),
    "trimestres"
)
print(
    "Tiempo, solo expansión:",
    round(tiempo_expansion, 2),
    "trimestres"
)

Comparación desde (BBB, Rec):

P(Default), economía mixta: 0.7835
P(Default), solo expansión: 0.6438
Tiempo, economía mixta: 10.86 trimestres
Tiempo, solo expansión: 16.94 trimestres


### ✅ Verificación con `jmarkov`

In [28]:
cadena_bono = dtmc(P_bono)
cadena_bono_expansion = dtmc(P_bono_expansion)

N_bono_jmarkov = np.zeros((6, 6))

for estado_inicial in range(6):
    for estado_visitado in range(6):

        valor = cadena_bono.absorbtion_times(
            start=estado_inicial,
            target=estado_visitado
        )

        N_bono_jmarkov[
            estado_inicial,
            estado_visitado
        ] = float(valor)

valor_default_original = (
    cadena_bono.absorbtion_probabilities(
        start=indice_BBB_Rec,
        target=6
    )
)

valor_default_expansion = (
    cadena_bono_expansion.absorbtion_probabilities(
        start=indice_BBB_Rec,
        target=6
    )
)

probabilidad_default_jmarkov = float(
    valor_default_original[0][0]
)

probabilidad_default_expansion_jmarkov = float(
    valor_default_expansion[0][0]
)

print("Fila de N desde (BBB, Rec), NumPy:")
print(
    np.round(
        matriz_fundamental_bono[indice_BBB_Rec, :],
        4
    )
)

print()
print("Fila de N desde (BBB, Rec), jmarkov:")
print(
    np.round(
        N_bono_jmarkov[indice_BBB_Rec, :],
        4
    )
)

print()
print(
    "P(Default) original, NumPy:",
    round(probabilidad_default_original, 4)
)
print(
    "P(Default) original, jmarkov:",
    round(probabilidad_default_jmarkov, 4)
)
print(
    "P(Default) expansión, NumPy:",
    round(probabilidad_default_expansion, 4)
)
print(
    "P(Default) expansión, jmarkov:",
    round(
        probabilidad_default_expansion_jmarkov,
        4
    )
)

Fila de N desde (BBB, Rec), NumPy:
[1.0929 0.4348 3.3299 2.8805 2.0863 1.0329]

Fila de N desde (BBB, Rec), jmarkov:
[1.0929 0.4348 3.3299 2.8805 2.0863 1.0329]

P(Default) original, NumPy: 0.7835
P(Default) original, jmarkov: 0.7835
P(Default) expansión, NumPy: 0.6438
P(Default) expansión, jmarkov: 0.6438


---
## Ejercicio integrador — Paciente con tratamiento y dos variables de estado

Un sistema de salud monitorea trimestralmente pacientes con una enfermedad crónica.

Las variables son:

- Severidad: Leve, Moderada o Grave;
- Adherencia: Alta o Baja.

Los estados transitorios son:

| Índice | Estado |
|---:|---|
| 0 | (Leve, Alta) |
| 1 | (Leve, Baja) |
| 2 | (Moderada, Alta) |
| 3 | (Moderada, Baja) |
| 4 | (Grave, Alta) |
| 5 | (Grave, Baja) |

Los estados absorbentes son:

- 6: Recuperado;
- 7: Salida.

Para que todos trabajen sobre el mismo modelo, utilizaremos las siguientes reglas fijas:

| Estado actual | Transiciones con probabilidad positiva |
|---|---|
| (L,A) | (L,A): 0.70; (L,B): 0.10; (M,A): 0.07; (M,B): 0.03; Recuperado: 0.10 |
| (L,B) | (L,A): 0.20; (L,B): 0.55; (M,A): 0.05; (M,B): 0.12; (G,B): 0.04; Recuperado: 0.04 |
| (M,A) | (L,A): 0.10; (M,A): 0.55; (M,B): 0.15; (G,A): 0.05; (G,B): 0.05; Recuperado: 0.10 |
| (M,B) | (L,B): 0.08; (M,A): 0.15; (M,B): 0.50; (G,A): 0.05; (G,B): 0.15; Recuperado: 0.02; Salida: 0.05 |
| (G,A) | (M,A): 0.10; (M,B): 0.05; (G,A): 0.45; (G,B): 0.20; Recuperado: 0.05; Salida: 0.15 |
| (G,B) | (M,B): 0.05; (G,A): 0.10; (G,B): 0.50; Salida: 0.35 |

### Tareas

1. Construya $P$ de tamaño $8\times8$.
2. Separe $\mathbf Q$ y $\mathbf R$.
3. Calcule $\mathbf N$ con la serie.
4. Calcule $\mathbf B=\mathbf N\mathbf R$.
5. Calcule el costo esperado acumulado del tratamiento antes de la absorción.
6. Verifique con `jmarkov`.

Los costos trimestrales, en millones de COP, son:

$$
[200,\;200,\;500,\;500,\;1200,\;1200].
$$

### Checkpoints antes de aceptar código generado por IA

Antes de ejecutar una solución completa, compruebe estos valores:

1. La fila de `(L,A)` debe ser:

```python
[0.70, 0.10, 0.07, 0.03, 0.00, 0.00, 0.10, 0.00]
```

2. La fila de `(M,B)` debe ser:

```python
[0.00, 0.08, 0.15, 0.50, 0.05, 0.15, 0.02, 0.05]
```

3. Cada fila debe sumar 1.

4. Después de construir `Q`, compare la primera fila de `Q @ Q` con el valor de referencia calculado en la siguiente celda.

In [ ]:
# Referencia para verificar Q^2.
# Ejecute esta celda después de construir P_paciente y Q_paciente.

fila_Q2_referencia = np.array([
    0.5170,
    0.1274,
    0.0970,
    0.0585,
    0.0050,
    0.0120
])

print("Fila de referencia de Q^2 desde (L,A):")
print(fila_Q2_referencia)

### Prompt sugerido para una IA

El prompt debe exigir que la IA utilice únicamente herramientas ya trabajadas: arreglos de NumPy, ciclos, condicionales, funciones sencillas y productos matriciales.

```text
Estoy resolviendo una cadena de Markov absorbente con 8 estados:

0=(L,A), 1=(L,B), 2=(M,A), 3=(M,B),
4=(G,A), 5=(G,B), 6=Recuperado, 7=Salida.

Usa exactamente las reglas de transición de la tabla del cuaderno.

Necesito que, en este orden:

1. Construyas P como un arreglo de NumPy y verifiques cada fila
   con un ciclo range.
2. Separes Q=P[:6,:6] y R=P[:6,6:].
3. Calcules N=I+Q+Q^2+... con un ciclo hasta que
   max(abs(Q^n)) sea menor que la tolerancia.
4. Calcules B=N@R.
5. Calcules el costo acumulado esperado usando ciclos,
   con costos [200,200,500,500,1200,1200].
6. Verifiques N y B con jmarkov únicamente al final.

No uses enumerate, zip, comprensiones de listas,
expresiones ternarias, axis=1, join ni generadores dentro de sum.
Usa nombres de variables descriptivos.
```

In [ ]:
# Construya aquí su solución o pegue el código generado por la IA.
# Compare primero las filas de P y la fila de Q @ Q
# con los checkpoints anteriores.

---
<small>Universidad de los Andes · Departamento de Ingeniería Industrial · Modelos de Decisión en el Tiempo</small>